## 4. División del conjunto de datos y transformaciones

In [28]:
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils.class_weight import compute_class_weight


# Paths
refined_input = Path("../data/processed/refined_data.parquet")
refined_input_N = Path("../data/processed/refined_data_N.parquet")

# Add src/ to Python path
sys.path.append(str(Path("../src").resolve()))
from data.var_type import split_symbolic_continuous

#### 1) Revisión general

In [23]:
df = pd.read_parquet(refined_input)
dn = pd.read_parquet(refined_input_N)

Primero de todo encuentro las variables numéricas y las categóricas/simbólicas para ambos dataset

In [24]:
symbolic, continuous = split_symbolic_continuous(df)
symbolicN, continuousN = split_symbolic_continuous(dn)

# For dn are the same because the only diff is we add another numeric variables so, symbolic = symbolicN
for col in symbolic:
    print(col, sorted(df[col].unique())[:10])

Protocol [np.int64(0), np.int64(6), np.int64(17)]
Fwd PSH Flags [np.uint32(0), np.uint32(1)]
Fwd URG Flags [np.uint32(0), np.uint32(1)]
FIN Flag Cnt [np.uint32(0), np.uint32(1)]
SYN Flag Cnt [np.uint32(0), np.uint32(1)]
RST Flag Cnt [np.uint32(0), np.uint32(1)]
PSH Flag Cnt [np.uint32(0), np.uint32(1)]
ACK Flag Cnt [np.uint32(0), np.uint32(1)]
URG Flag Cnt [np.uint32(0), np.uint32(1)]
CWE Flag Count [np.uint32(0), np.uint32(1)]
ECE Flag Cnt [np.uint32(0), np.uint32(1)]
Label ['Benign', 'DDOS attack-HOIC', 'DDOS attack-LOIC-UDP', 'DoS attacks-GoldenEye', 'DoS attacks-Slowloris', 'FTP-BruteForce', 'Infilteration', 'SSH-Bruteforce']
Dst Port Cat ['dfs', 'ephemeral', 'ftp', 'registered', 'ssh', 'web', 'well_known_other']
attack_group ['Benign', 'Bruteforce', 'DDOS', 'DoS', 'Infiltration']
attack_or_benign ['Attack', 'Benign']


#### 2) Codificación de las variables simbólicas

Primero divido el conjunto de variables simbólicas en sus diferentes grupos y los codifico acorde a su tipo.

In [25]:
# Symbolic var config
target_col = "attack_or_benign"   # or "Label" or "attack_group"
binary_cols = [
    "Fwd PSH Flags", "Fwd URG Flags",
    "FIN Flag Cnt", "SYN Flag Cnt",
    "RST Flag Cnt", "PSH Flag Cnt",
    "ACK Flag Cnt", "URG Flag Cnt",
    "CWE Flag Count", "ECE Flag Cnt"
]
categorical_cols = ["Protocol", "Dst Port Cat"]

Preparo los conjuntos de variables y respuesta.

In [26]:
# Features and labels
X = df.drop(columns=['Label', 'attack_group', 'attack_or_benign'])
y = df[target_col] #I'll start with the binary one which is the easiest
Xn = dn.drop(columns=['Label', 'attack_group', 'attack_or_benign'])
yn = dn[target_col] #I'll start with the binary one which is the easiest

Ahora hago la codificación:

In [27]:
# Cast binary columns
for col in binary_cols:
    X[col] = X[col].astype("uint8")
    Xn[col] = Xn[col].astype("uint8")

# One-hot encode categoricals
X = pd.get_dummies(X, columns=categorical_cols)
Xn = pd.get_dummies(Xn, columns=categorical_cols)

#### 3) División del conjunto de datos

Ahora procedo a dividir el conjunto en un conjunto de train (80%) y uno de test (20%). Luego validaremos los modelos por validación cruzada por lo que no es necesario hacer un conjunto aparte de validación. Además continuó con la codificación de la variable respuesta.

In [17]:
# df Train (80%) / Test (20%)
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42,
    shuffle=True
)

# dn Train (80%) / Test (20%)
X_trainN, X_testN, y_trainN, y_testN = train_test_split(
    Xn,
    yn,
    test_size=0.20,
    stratify=yn,
    random_state=42,
    shuffle=True
)

# Codificación de la variable respuesta


# Shapes
print("Dataset df:")
print("  Train shape:", X_train.shape, y_train.shape)
print("  Test shape:", X_test.shape, y_test.shape)

print("Dataset dn:")
print("  Train shape:", X_trainN.shape, y_trainN.shape)
print("  Test shape:", X_testN.shape, y_testN.shape)

Dataset df:
  Train shape: (2573108, 79) (2573108,)
  Test shape: (643278, 79) (643278,)
Dataset dn:
  Train shape: (1724336, 80) (1724336,)
  Test shape: (431085, 80) (431085,)


#### 4) Estandarizado de los datos

Para evitar sesgos hago el estandarizado por separado.

In [20]:
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train[continuous])
X_test_scaled = scaler.transform(X_test[continuous])

In [21]:
scaler_dn = RobustScaler()
X_trainN_scaled = scaler_dn.fit_transform(X_trainN[continuousN])
X_testN_scaled = scaler_dn.transform(X_testN[continuousN])

#### 5) Balanceo del conjunto de train 

A la hora de tratar con el problema de clases desbalanceadas opto por la opción de usar pesos. Para los modelos que he elegido XGBoost y Rand Forest las opciones para balancear vienen practicamente incluidas, sólo en XGBoost debo obetner los pesos primero.

**XGBoost**

In [ ]:
weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
)

sample_weights = weights[y_train]
xgb.fit(X_train, y_train, sample_weight=sample_weights)

**Random Forest**

In [ ]:
rf = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight="balanced"
)

rf.fit(X_train, y_train)